In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency

import sys
import os

sys.path.append(
    os.path.abspath("..")
)

from src.hypothesis_tests import (
    run_ttest,
    run_chi2
)

In [3]:
df = pd.read_csv(
    "../data/insurance_data.txt",
    sep="|",
    low_memory=False
)

In [4]:
df["HasClaim"] = np.where(
    df["TotalClaims"] > 0,
    1,
    0
)

In [5]:
df["LossRatio"] = np.where(
    df["TotalPremium"] > 0,
    df["TotalClaims"] / df["TotalPremium"],
    np.nan
)

In [6]:
df["Margin"] = (
    df["TotalPremium"] -
    df["TotalClaims"]
)

H₀: There are no risk differences across provinces.

In [7]:
group_a = df[
    df["Province"] == "Gauteng"
]["LossRatio"]

group_b = df[
    df["Province"] == "Western Cape"
]["LossRatio"]

In [8]:
province_test = run_ttest(
    group_a,
    group_b
)

province_test

{'test': 'Independent t-test', 'p_value': np.float64(0.03883632811925487)}

Reject H₀

Meaning:

There IS statistically significant evidence of a risk difference between the two provinces you compared.

In [9]:
group_a.mean(), group_b.mean()

(np.float64(0.4289266807380461), np.float64(0.34181035714327007))


### Results
The independent t-test produced a p-value of 0.0388, which is below the 0.05 significance threshold.

Average loss ratios:
- Group A Province: 0.4289
- Group B Province: 0.3418

### Decision
Reject H₀.

### Business Interpretation
The analysis provides statistically significant evidence that risk levels differ between the two provinces. The first province exhibits approximately 8.7 percentage points higher average loss ratios compared to the second province.

This suggests that province-level pricing adjustments or regional underwriting strategies may improve portfolio profitability and risk segmentation accuracy.

In [10]:
df["PostalCode"].value_counts().head(10)

PostalCode
2000    133498
122      49171
7784     28585
299      25546
7405     18518
458      13775
8000     11794
2196     11048
470      10226
7100     10161
Name: count, dtype: int64

In [11]:
zip_a = df[
    df["PostalCode"] == 7784
]["TotalClaims"].dropna()

zip_b = df[
    df["PostalCode"] == 7405
]["TotalClaims"].dropna()

In [12]:
zip_test = run_ttest(
    zip_a,
    zip_b
)

zip_test

{'test': 'Independent t-test', 'p_value': np.float64(0.18013217703189774)}

In [13]:
zip_a.mean(), zip_b.mean()

(np.float64(61.494933896749934), np.float64(32.89008844879237))

## Hypothesis Test 2: Zip Code Risk Differences

### Null Hypothesis
H₀: There are no risk differences between zip codes.

### KPI Selected
Claim Severity (Average TotalClaims)

### Statistical Test
Independent two-sample t-test

### Results
The independent t-test produced a p-value of 0.1801, which exceeds the 0.05 significance threshold.

Average claim severity:
- Postal Code 7784: 61.49
- Postal Code 7405: 32.89

### Decision
Fail to reject H₀.

### Business Interpretation
Although the average claim severity appears numerically higher in postal code 7784, the observed difference is not statistically significant at the 5% significance level.

This suggests that the variation in claims between the two postal codes may be driven by randomness or high claim variability rather than a consistent underlying risk difference. Therefore, the current evidence does not strongly support implementing zip-code-specific pricing adjustments based solely on this comparison.

In [14]:
margin_a = df[
    df["PostalCode"] == 7784
]["Margin"].dropna()

margin_b = df[
    df["PostalCode"] == 7405
]["Margin"].dropna()

In [15]:
margin_test = run_ttest(
    margin_a,
    margin_b
)

margin_test

{'test': 'Independent t-test', 'p_value': np.float64(0.14519862534780517)}

In [16]:
margin_a.mean(), margin_b.mean()

(np.float64(-13.47124236869368), np.float64(17.51682453042594))

## Hypothesis Test 3: Margin Differences Between Zip Codes

### Null Hypothesis
H₀: There is no significant margin difference between zip codes.

### KPI Selected
Margin (TotalPremium − TotalClaims)

### Statistical Test
Independent two-sample t-test

### Results
The independent t-test produced a p-value of 0.1452, which is greater than the 0.05 significance threshold.

Average margins:
- Postal Code 7784: -13.47
- Postal Code 7405: 17.52

### Decision
Fail to reject H₀.

### Business Interpretation
Although the two postal codes exhibit noticeably different average profitability levels, the observed difference is not statistically significant at the 5% level.

Postal code 7784 appears unprofitable on average, while postal code 7405 shows positive average margins. However, high variability in claims and premiums likely contributes to uncertainty in the estimates.

The current evidence is insufficient to justify zip-code-specific pricing adjustments solely based on margin differences.

In [17]:
df["HasClaim"] = np.where(
    df["TotalClaims"] > 0,
    1,
    0
)

In [18]:
gender_table = pd.crosstab(
    df["Gender"],
    df["HasClaim"]
)

gender_table

HasClaim,0,1
Gender,,
Female,6741,14
Male,42723,94
Not specified,938324,2666


In [19]:
gender_test = run_chi2(
    gender_table
)

gender_test

{'test': 'Chi-squared', 'p_value': np.float64(0.026570248768437145)}

In [20]:
if gender_test["p_value"] < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


In [21]:
df.groupby("Gender")["HasClaim"].mean()

Gender
Female           0.002073
Male             0.002195
Not specified    0.002833
Name: HasClaim, dtype: float64

## Hypothesis Test 4: Gender-Based Risk Differences

### Null Hypothesis
H₀: There is no significant risk difference between Women and Men.

### KPI Selected
Claim Frequency (proportion of policies with at least one claim)

### Statistical Test
Chi-squared test of independence

### Results
The chi-squared test produced a p-value of 0.0266, which is below the 0.05 significance threshold.

Observed claim frequencies:
- Female: 0.00207
- Male: 0.00220
- Not specified: 0.00283

### Decision
Reject H₀.

### Business Interpretation
The analysis indicates statistically significant differences in claim frequency across gender categories. However, the observed differences in claim frequencies are extremely small in practical terms.

While the large dataset size allows these differences to be detected statistically, the actual business impact may be limited. Therefore, gender alone may not justify substantial pricing adjustments without additional supporting risk factors.

In [22]:
results = pd.DataFrame([

    {
        "Hypothesis":
            "Province Risk Difference",

        "KPI":
            "Loss Ratio",

        "Test":
            "Independent t-test",

        "P-Value":
            province_test["p_value"],

        "Decision":
            (
                "Reject H0"
                if province_test["p_value"] < 0.05
                else "Fail to Reject H0"
            )
    },

    {
        "Hypothesis":
            "Zip Code Risk Difference",

        "KPI":
            "Claim Severity",

        "Test":
            "Independent t-test",

        "P-Value":
            zip_test["p_value"],

        "Decision":
            (
                "Reject H0"
                if zip_test["p_value"] < 0.05
                else "Fail to Reject H0"
            )
    },

    {
        "Hypothesis":
            "Margin Difference by Zip Code",

        "KPI":
            "Margin",

        "Test":
            "Independent t-test",

        "P-Value":
            margin_test["p_value"],

        "Decision":
            (
                "Reject H0"
                if margin_test["p_value"] < 0.05
                else "Fail to Reject H0"
            )
    },

    {
        "Hypothesis":
            "Gender Risk Difference",

        "KPI":
            "Claim Frequency",

        "Test":
            "Chi-squared",

        "P-Value":
            gender_test["p_value"],

        "Decision":
            (
                "Reject H0"
                if gender_test["p_value"] < 0.05
                else "Fail to Reject H0"
            )
    }

])

results

,Hypothesis,KPI,Test,P-Value,Decision
0,Province Risk Difference,Loss Ratio,Independent t-test,0.038836,Reject H0
1,Zip Code Risk Difference,Claim Severity,Independent t-test,0.180132,Fail to Reject H0
2,Margin Difference by Zip Code,Margin,Independent t-test,0.145199,Fail to Reject H0
3,Gender Risk Difference,Claim Frequency,Chi-squared,0.026570,Reject H0


# Final Conclusions

The hypothesis testing analysis evaluated whether statistically significant differences exist across several insurance risk dimensions.

Key findings include:

- Province-level risk differences were statistically significant, suggesting that geographic pricing adjustments may improve underwriting performance.
- Zip-code-level differences in both claim severity and profitability were not statistically significant, indicating that localized pricing adjustments may not currently be justified.
- Gender-based differences in claim frequency were statistically significant; however, the practical effect size was relatively small.

Overall, the results demonstrate that broader geographic segmentation appears more meaningful than highly granular segmentation for this portfolio. The findings provide a statistically grounded foundation for future predictive modeling and pricing optimization tasks.

# Business Recommendations

## 1. Province-Based Pricing Adjustments

The province-level analysis revealed statistically significant differences in loss ratios between regions. This suggests that geographic location is an important driver of insurance risk.

Recommendation:
- Consider implementing province-specific pricing adjustments or underwriting rules.
- Higher-risk provinces may require increased premiums, stricter underwriting criteria, or enhanced fraud monitoring.
- Lower-risk provinces may provide opportunities for more competitive pricing strategies to improve customer acquisition.

---

## 2. Gender-Based Risk Segmentation

The gender analysis identified statistically significant differences in claim frequency across customer groups. However, the practical effect size was relatively small.

Recommendation:
- Gender alone should not be used as a primary pricing factor.
- Instead, gender may be considered alongside stronger predictive variables such as vehicle type, driving history, geographic location, and claim history.
- Further modeling should evaluate whether gender meaningfully improves predictive performance in combination with other variables.

---

## 3. Cautious Use of Zip-Code-Level Segmentation

The analysis did not identify statistically significant differences in either claim severity or profitability between the selected postal codes.

Recommendation:
- Avoid overly granular zip-code-level pricing adjustments based solely on the current evidence.
- Additional data or broader geographic aggregation may be necessary before implementing location-specific pricing strategies at the postal-code level.